In [1]:
import os, torch
from google.colab import drive
from transformers import AutoTokenizer, GPT2LMHeadModel

# load model from drive checkpoint
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

MODEL_DIR = "/content/drive/MyDrive/recipe_gpt2/model_pretrained_250k_ingfirst"
BOS, EOS = "<|startofrecipe|>", "<|endofrecipe|>"

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = GPT2LMHeadModel.from_pretrained(MODEL_DIR)
model.eval()
if torch.cuda.is_available():
    model.to("cuda")
print("Model loaded from", MODEL_DIR)

def make_recipe(ingredients, temperature=0.7, top_p=0.9, max_length=512):
    # Sort + lowercase to match training
    norm = ", ".join(sorted(i.strip().lower() for i in ingredients.split(",") if i.strip()))
    prompt = f"{BOS}<|ingredients|>{norm}<|title|>"   # ends at title -> model writes the rest
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            inputs["input_ids"], attention_mask=inputs["attention_mask"],
            max_length=max_length, do_sample=True, temperature=temperature, top_p=top_p,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.convert_tokens_to_ids(EOS),
        )
    text = tokenizer.decode(out[0], skip_special_tokens=False)
    print(text.replace(BOS, "").replace(EOS, "").replace("<|pad|>", "")
              .replace("<|ingredients|>", "INGREDIENTS: ")
              .replace("<|title|>", "\nTITLE: ")
              .replace("<|directions|>", "\nDIRECTIONS: ").strip())

Mounted at /content/drive


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Model loaded from /content/drive/MyDrive/recipe_gpt2/model_pretrained_250k_ingfirst


In [2]:
make_recipe("chicken, mango, lime, tortillas")

INGREDIENTS: chicken, lime, mango, tortillas
TITLE: Chicken Wrap
DIRECTIONS: Take chicken and roll in lime. Wrap in tortillas and eat.


In [3]:
make_recipe("chicken, beef, pork")

INGREDIENTS: beef, chicken, pork
TITLE: Chicken And Pork
DIRECTIONS: Take a large skillet and put the beef, pork and pork in the pan. Add the water, stir and cook over medium heat until it is tender and the meat is cooked through. You can serve this over rice.


In [4]:
make_recipe("bacon, chili powder, chocolate, maple syrup, milk, sugar, watermelon")

INGREDIENTS: bacon, chili powder, chocolate, maple syrup, milk, sugar, watermelon
TITLE: Chili
DIRECTIONS: Boil watermelon and syrup for 2 minutes. Drain and set aside. Mix maple syrup with sugar and watermelon and stir. Add salt and pepper to taste. Put in freezer for 1 hour.


In [5]:
make_recipe("rice, tofu, eggs, salt, pepper")

INGREDIENTS: eggs, pepper, rice, salt, tofu
TITLE: Tofu Rice
DIRECTIONS: Put the tofu in the microwave and microwave on high for about 3 minutes. When it's done, add the egg yolks and cook on high for about 5 minutes. When it's done, put it in the fridge and stir for another 5 minutes. Remove it from the microwave and add the salt and pepper. Serve it with a dollop of rice and a dollop of tofu.


In [6]:
make_recipe("shrimp, lettuce, tomato, cucumber, rice, syrup")

INGREDIENTS: cucumber, lettuce, rice, shrimp, syrup, tomato
TITLE: Shrimp Salad
DIRECTIONS: Peel and chop the shrimp. Add the ingredients and mix well. Serve in bowls.
